In [1]:
import numpy as np

In [2]:
import cupy as cp

ModuleNotFoundError: No module named 'cupy'

In [3]:
!nvidia-smi

Wed Jul  8 07:26:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     On  |   00000000:01:00.0 Off |                    0 |
|  0%   31C    P8             22W /  150W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
pip install cupy-cuda13x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 MB 190.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cupy-cuda13x] [cupy-cuda13x]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import cupy as cp

In [6]:
pip install cuda-cccl[cu13]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 70.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 248.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 265.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 317.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [cuda-cccl]/5 [cuda-cccl]]gs]
Note: you may need to restart the kernel to use updated packages.


CuPy was explicitly designed as a drop-in replacement for NumPy to run on NVIDIA GPUs (via CUDA) and AMD GPUs (via ROCm), the API is nearly identical—you just swap np for cp.

### 1. The Basics & GPU Allocation

In [9]:
import cupy as cp

# 1. Initialize real CuPy arrays on the GPU
arr = cp.array([[1, 2], [3, 4]])
a = cp.array([1, 2, 3])
b = cp.array([4, 5, 6])

# 2. Reshaping operations (modifies GPU memory addressing views)
print("Flattened GPU Array:", arr.ravel())
print("Reshaped GPU Array:\n", arr.reshape(4, 1))
print("Transposed GPU Matrix:\n", arr.T)

# 3. Combining & Stacking directly on the GPU
print("Vertical Stack:\n", cp.vstack((a, b)))
print("Horizontal Stack:", cp.hstack((a, b)))
print("Concatenated:", cp.concatenate((a, b), axis=0))

Flattened GPU Array: [1 2 3 4]
Reshaped GPU Array:
 [[1]
 [2]
 [3]
 [4]]
Transposed GPU Matrix:
 [[1 3]
 [2 4]]
Vertical Stack:
 [[1 2 3]
 [4 5 6]]
Horizontal Stack: [1 2 3 4 5 6]
Concatenated: [1 2 3 4 5 6]


In [3]:
import cupy as cp

# Allocate standard 1D and 2D arrays directly on GPU memory
arr_1d = cp.array([1, 2, 3])
arr_2d = cp.array([(1, 2, 3), (4, 5, 6)], dtype=cp.float32)  # float32 is optimal for GPUs

# GPU Placeholders & Sequence Generators
cp.zeros((3, 4))            # 3x4 array of zeros on GPU
cp.ones((2, 3, 4))          # Array of ones on GPU
cp.arange(10, 25, 5)        # Range array: [10, 15, 20]
cp.linspace(0, 2, 9)        # 9 evenly spaced values between 0 and 2
cp.eye(3)                   # 3x3 identity matrix on GPU
cp.random.random((2, 2));   # 2x2 array of random floats using GPU random generators

### 2. Moving Data Between CPU (NumPy) & GPU (CuPy)
Data transfer over the PCIe bus is expensive. Minimize these transfers in your production pipelines.

In [4]:
import numpy as np
import cupy as cp

# 1. CPU to GPU (Host to Device)
x_cpu = np.array([1, 2, 3])
x_gpu = cp.asarray(x_cpu)    # Moves data to GPU memory (or returns x_cpu if already on GPU)

# 2. GPU to CPU (Device to Host)
y_gpu = cp.array([4, 5, 6])
y_cpu = cp.asnumpy(y_gpu)    # Pulls data back to CPU host memory
# Alternatively:
y_cpu = y_gpu.get()          # Explicitly copies data back to a NumPy array

### 3. Array Manipulation (Reshaping & Stacking)
These operations modify how the GPU memory addresses the array layout, usually without copying the underlying raw data payload.

In [11]:
# Reshaping
arr.ravel()             # Flatten a multi-dimensional array to 1D
arr.reshape(4, 1)       # Change shape
arr.T                   # Transpose

# Combining & Stacking on GPU
cp.vstack((a, b))       # Stack arrays vertically (row-wise)
cp.hstack((a, b))       # Stack arrays horizontally (column-wise)
cp.concatenate((a, b), axis=0);

#### 💡 CuPy Memory Pro-Tip:
When you perform operations like .ravel(), .reshape(), or .T in CuPy, the GPU doesn't waste time copying or moving data around in its VRAM. Instead, it instantly creates a view of the existing array, simply changing how the GPU threads read the underlying memory addresses!

### 4. Indexing & Slicing
Slicing operations on CuPy arrays create views, not copies, directly inside GPU device memory.

In [13]:
import cupy as cp

# 1. Properly define 'a' as a 1D vector and 'b' as a 2D matrix on the GPU
a = cp.array([10, 20, 30, 40])
b = cp.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])  # 2D matrix (3 rows, 3 columns)

# 2. Now these GPU slicing operations will execute perfectly!
print("Rows 0-1, Column 1:", b[0:2, 1])  # Yields: [2, 5]
print("All rows, Column 0:", b[:, 0])  # Yields: [1, 4, 7]

# 3. Advanced Masking (Triggers parallel GPU kernel execution)
print("GPU Mask Filter:", a[a < 30])  # Yields: [10, 20]

Rows 0-1, Column 1: [2 5]
All rows, Column 0: [1 4 7]
GPU Mask Filter: [10 20]


In [14]:
# 2D Slicing (matrix[row, column])
b[0:2, 1]    # Select items from rows 0 and 1 in column 1
b[:, 0]      # Select every item in column 0

# Advanced Indexing (Triggers GPU kernel execution)
a[a < 2]     # Boolean Masking: Select elements from a less than 2
b[[1, 0], [0, 1]];  # Fancy Indexing

#### 💡 Why CuPy Views are Faster than NumPy Views
When you perform basic slicing on a 2D array in CuPy (like b[:, 0]), CuPy calculates the memory stride offsets instantly and points to the data directly in the GPU's VRAM. This avoids passing layout metadata back and forth across the slow PCIe bus to the CPU, keeping your data pipeline localized entirely on the graphics hardware.

### 5. Fast Mathematical Operations & Reductions
Operations are executed in parallel across thousands of GPU stream cores using compiled CUDA/ROCm kernels.

In [16]:
import cupy as cp

# 1. Define shapes that CAN be broadcast together
# Let's make 'b' a 2x3 matrix, and 'a' a 1x3 row vector
b = cp.array([[10, 20, 30], [40, 50, 60]])  # Shape: (2, 3)
a = cp.array([1, 2, 3])  # Shape: (3,)

# 2. Now the GPU element-wise math arithmetic runs flawlessly!
# CuPy automatically broadcasts 'a' vertically across both rows of 'b'
print("Addition:\n", cp.add(a, b))  # Or: a + b
print("Subtraction:\n", cp.subtract(b, a))  # Or: b - a
print("Multiplication:\n", cp.multiply(a, b))  # Or: a * b

# 3. Standard element-wise transforms
print("Square Root of a:", cp.sqrt(a))
print("Exponential of a:", cp.exp(a))

# 4. GPU-Accelerated Reductions
print("Grand Total of matrix b:", b.sum())
print("Column-wise Maximums of b:", b.max(axis=0))  # Yields: [40, 50, 60]
print("Row-wise Cumulative Sum of b:\n", b.cumsum(axis=1))

Addition:
 [[11 22 33]
 [41 52 63]]
Subtraction:
 [[ 9 18 27]
 [39 48 57]]
Multiplication:
 [[ 10  40  90]
 [ 40 100 180]]
Square Root of a: [1.         1.41421356 1.73205081]
Exponential of a: [ 2.71828183  7.3890561  20.08553692]
Grand Total of matrix b: 210
Column-wise Maximums of b: [40 50 60]
Row-wise Cumulative Sum of b:
 [[ 10  30  60]
 [ 40  90 150]]


In [19]:
# Element-wise Arithmetic Operations (Parallel GPU Execution)
cp.add(a, b)         # Alternatively: a + b
cp.subtract(a, b)    # Alternatively: a - b
cp.multiply(a, b)    # Alternatively: a * b
cp.sqrt(a)           # Square root of each element
cp.exp(a)            # Exponentiation (e^x)

# GPU-Accelerated Reductions
a.sum()              # Array-wise sum
b.max(axis=0)        # Maximum value down columns
b.cumsum(axis=1)     # Cumulative sum across rows
a.mean()             # Arithmetic Mean
cp.median(b);        # Median

#### 💡 The Golden Rules of GPU Broadcasting:
For two arrays to broadcast together, compare their shapes starting from the **rightmost (trailing) dimension** and move left:
- Are the dimensions equal? (e.g., both are 3) $\rightarrow$ **Pass!**
- Is one of the dimensions 1? (e.g., matching a (2, 3) with a (1, 3)) $\rightarrow$ **Pass!** (The array with size 1 will be duplicated along that axis).

### 6. Writing Custom CUDA Kernels
When standard element-wise array operations aren't enough, CuPy lets you write raw C++ CUDA code directly inside your Python file:

In [20]:
# Define a custom element-wise CUDA kernel
squared_diff_kernel = cp.ElementwiseKernel(
    'float32 x, float32 y',  # Input argument types
    'float32 z',             # Output argument type
    'z = (x - y) * (x - y)', # Actual C++ CUDA code line
    'squared_diff_kernel'    # Name of the kernel function
)

x = cp.array([1, 2, 3], dtype=cp.float32)
y = cp.array([4, 5, 6], dtype=cp.float32)

z = squared_diff_kernel(x, y) # Executed instantly in parallel on the GPU!

## The 5-Minute CuPy Puzzle: "The GPU Memory Ghost"
Imagine you run the following sequence of operations in a Python environment with an active GPU:

In [21]:
import numpy as np
import cupy as cp

# 1. Initialize a standard array on the CPU
x_cpu = np.array([10, 20, 30, 40])

# 2. Push it to the GPU
x_gpu = cp.asarray(x_cpu)

# 3. Create a slice (view) of the GPU array
view_gpu = x_gpu[1:3]

# 4. Modify an element inside the GPU view
view_gpu[0] = 99

**Your Mission:**
Without running the code, predict exactly what elements are stored in these two arrays right now:

What is the current value of the original CPU array: `x_cpu`?

What is the current value of the GPU array: `x_gpu`?

<details>
<summary><b>Click to reveal the 5-Minute CuPy "The GPU Memory Ghost" Challenge</b></summary>
The Correct Answer:
    
- `x_cpu` is still: [10, 20, 30, 40] (Unchanged!)
- `x_gpu` is now: [10, 99, 30, 40] (Modified!)

#### Why this happens:
- The PCIe Moat: When you called cp.asarray(x_cpu), CuPy allocated a brand-new, isolated block of memory entirely inside the GPU's VRAM and copied the data across the PCIe bus. They no longer share a physical memory address. Modifying things on the GPU cannot automatically alter the CPU host memory.

- The Power of Views: When you sliced `x_gpu[1:3]`, CuPy did not copy the data inside the GPU. Instead, it created a structural view pointing directly to the original GPU array's internal VRAM slot.

- The Ripple Effect: Because view_gpu is a view, changing `view_gpu[0]` (which points to index 1 of the parent array) instantly alters the parent array `x_gpu` directly inside the graphics hardware.

To get the modified data back to your CPU array, you would have to explicitly bridge the gap yourself using:

```python
x_cpu = cp.asnumpy(x_gpu)
```
Did your mental model successfully separate the CPU host memory from the GPU device memory?

### The 10-Minute Challenge: "The Cosmic Ray Filter"

A satellite detector tracks high-energy physics events, but background cosmic ray noise corrupts the data matrix. You need to use the GPU to clean up the sensor grid swiftly.

In [26]:
# Setup the Simulation
import numpy as np
import cupy as cp

# A 4x4 matrix of raw sensor telemetry generated on the CPU
raw_telemetry = np.array([
    [101,  12,  85,  14],
    [ 22, 140,  19,  33],
    [ 90,  15, 112,  45],
    [  8,  52,  11, 135]
])

# Threshold criteria limit
noise_floor = 50

**The Mission**
Write the missing code lines to complete the filter pipeline entirely inside the GPU VRAM:

- Task A (Host-to-Device): Move the raw data matrix from the CPU over to the GPU memory. Name the new GPU variable data_gpu.

- Task B (Boolean Masking): Create a GPU mask that identifies all elements that are strictly greater than the noise_floor.

- Task C (In-Place Zeroing): Use that mask to target the background noise. Set every element below or equal to the noise floor to exactly 0 inside your original data_gpu array.

- Task D (Device-to-Host): Pull the cleaned matrix back to the CPU as a standard NumPy array named clean_telemetry.

Expected Array Output:
```python
[[101   0  85   0]
 [  0 140   0   0]
 [ 90   0 112   0]
 [  0   0   0 135]]

<details>
<summary><b>Click to reveal the 10-Minute CuPy GPU Master Challenge</b></summary>

```python
# Task A: Explicitly cross the PCIe bus to populate GPU memory
data_gpu = cp.asarray(raw_telemetry)

# Task B: Generate a matrix of boolean toggles directly on the GPU hardware
mask = data_gpu > noise_floor

# Task C: Invert the mask logic (~mask) to flag background noise, 
# then selectively overwrite those spots in-place without altering other elements
data_gpu[~mask] = 0

# Task D: Bring the finalized matrix back across the PCIe bus to host memory
clean_telemetry = cp.asnumpy(data_gpu)

print("Cleaned GPU Matrix Matrix:\n", data_gpu)
print("Final Output Type:", type(clean_telemetry))
```

#### 💡 Core Architectural Concepts:
- The Memory Boundary: The data must be explicitly pushed to the hardware (`cp.asarray`) and explicitly pulled back (`cp.asnumpy`) to sync the host CPU with the device GPU.

- In-Place Mutation vs. Filtering: Instead of pulling matching values out into a flat 1D array (which happens if you run data_gpu[mask]), using the mask as an index assignment target (data_gpu[~mask] = 0) mutates the data in-place, preserving the matrix shape.

- The Bitwise NOT (~) Operator: Fast practice using logical bitwise negation to turn a "keep" mask into a "discard" mask efficiently inside GPU kernels.


### CuPy gotchas and sharp ends
While CuPy is an incredibly powerful "drop-in" replacement for NumPy, treating it exactly like NumPy will eventually lead to major performance degradation, silent bugs, or hard crashes.

Because you are writing Python code that orchestrates heavy-duty asynchronous C++/CUDA kernels across a hardware bus (PCIe), there are several critical architectural "sharp edges" you need to be aware of.

#### 1. The PCIe "Tax" (Implicit Data Transfer)
The single biggest performance killer in CuPy isn't the GPU computation time—it's accidentally moving data back and forth between the CPU (Host) and GPU (Device).

In [29]:
import cupy as cp
import numpy as np

x_gpu = cp.random.random((10000, 10000))

# ❌ HUGE GOTCHA: Mixing CuPy and NumPy triggers silent, slow data transfers
result = np.sin(x_gpu)  # Triggers a slow copy back to CPU to let NumPy process it!

# ❌ GOTCHA: Accessing individual elements or printing in a loop pulls data across PCIe
for i in range(10):
    print(
        x_gpu[i, 0]
    )  # Every single iteration forces the CPU to wait for the GPU over the PCIe bus

0.6486791278189905
0.5876849895417715
0.15285404057996116
0.4892129736595751
0.598813725338577
0.5229149327782336
0.6164014364101404
0.446335997765624
0.2815002034486153
0.7729524517306048


**The Fix:**
Keep your data pipeline entirely on the GPU. If you must bring data back to the CPU, do it exactly once at the very end of your script using cp.asnumpy(x_gpu) or x_gpu.get().

#### 2. Asynchronous Execution & False Benchmarks
CuPy operations are asynchronous. When you call a function like cp.dot(A, B), the CPU simply submits the task to the GPU's execution queue and instantly moves to the next line of Python code before the GPU finishes computing.

In [33]:
import time

A = cp.random.random((5000, 5000), dtype=cp.float32)
B = cp.random.random((5000, 5000), dtype=cp.float32)

# ❌ FALSE BENCHMARK: This does NOT measure actual GPU computation time!
start = time.time()
y_gpu = cp.matmul(A, B)
print(f"Time taken: {time.time() - start} seconds")  # Way too fast to be true!

Time taken: 0.0005593299865722656 seconds


**The Fix:**
If you are profiling or benchmarking code, you must explicitly force the CPU to wait for the GPU kernel queue to completely clear using `cp.cuda.Stream.null.synchronize()` before stopping your timer.

In [37]:
#  ACCURATE BENCHMARK
cp.cuda.Stream.null.synchronize()  # Clear queue before starting
start = time.time()

y_gpu = cp.matmul(A, B)

cp.cuda.Stream.null.synchronize()  # Force CPU to wait for the GPU to finish
print(f"Actual GPU Time: {time.time() - start} seconds")

Actual GPU Time: 0.014340877532958984 seconds


#### 3. The `dtype` Precision Trap (Float64 vs Float32)
Python and NumPy default to 64-bit floating-point numbers (`float64`) for almost everything. However, consumer GPUs (like NVIDIA GeForce cards) are physically optimized for 32-bit (`float32`) or 16-bit precision.

In [38]:
# ❌ SHARP EDGE: Inheriting float64 from a Python list
x_gpu = cp.array([1.0, 2.0, 3.0])  # Defaults to float64! 
# This runs significantly slower and consumes double the precious VRAM.

**The Fix:**
Get into the habit of explicitly declaring `dtype=cp.float32` whenever you initialize arrays on a GPU unless your scientific calculation strictly requires double precision.

In [39]:
x_gpu = cp.array([1.0, 2.0, 3.0], dtype=cp.float32)  # Optimal GPU performance

#### 4. In-Place Modifications & Graph Memory Fragmentation
Because GPU memory allocation is an expensive hardware operation, CuPy uses an internal memory pool to recycle VRAM blocks. However, creating large intermediate arrays or slicing arrays haphazardly can fragment this pool, leading to sudden `Out of Memory (OOM)` errors even if your data should technically fit.

In [41]:
# ❌ FRAGMENTATION RISK: Creating unnecessary intermediate arrays
A = cp.random.random((5000, 5000), dtype=cp.float32)
B = cp.random.random((5000, 5000), dtype=cp.float32)

C = A + B  # Allocates a brand-new 5000x5000 block in VRAM

**The Fix:**
Whenever possible, utilize in-place operations or the `out` parameter to reuse pre-allocated blocks of memory directly.

In [42]:
#  VRAM EFFICIENT: Overwrites the memory block of A directly
A += B 

# Or using the out parameter:
cp.add(A, B, out=A)

array([[2.5058653 , 0.513376  , 0.2848227 , ..., 1.823576  , 1.7717321 ,
        1.7067153 ],
       [1.4588501 , 1.9317528 , 1.976846  , ..., 1.1744112 , 1.195846  ,
        1.926295  ],
       [1.4204168 , 0.5922106 , 1.0089262 , ..., 0.71111965, 1.282002  ,
        0.71727663],
       ...,
       [1.003829  , 2.03791   , 1.3589848 , ..., 0.9107641 , 0.9823497 ,
        0.5137649 ],
       [1.5529618 , 1.4440935 , 1.5931349 , ..., 0.5936155 , 1.7908819 ,
        1.9252366 ],
       [1.2608688 , 0.5370942 , 1.9241627 , ..., 0.95213276, 0.90183616,
        0.39658844]], shape=(5000, 5000), dtype=float32)

#### 5. CPU Fallbacks Aren't Always Implemented
While CuPy covers about 95% of the standard NumPy API, it doesn't support everything. If you call an obscure linear algebra function or specific polynomial routine that CuPy hasn't written a custom CUDA kernel for, your code will crash with an `AttributeError`.

**The Fix:**
If you are writing library code that needs to work seamlessly whether a user has a GPU or not, use CuPy’s fallback mechanism (`get_array_module`) to write backend-agnostic code:

In [43]:
import cupy as cp
import numpy as np

def compute_pipeline(x):
    # Dynamically detects whether 'x' lives on the CPU or GPU
    xp = cp.get_array_module(x)
    
    # 'xp' becomes 'cp' if x is on GPU, or 'np' if x is on CPU
    return xp.sin(x) + 2